In [6]:
import pandas as pd
import numpy as np
URL = "../data/numeros_tva.csv"
df = pd.read_csv(URL)

- Combien de pays distincts ? 
    Il y a 15 pays distinct.
- Combien de formats différents pour un même pays ? 
    Dont 2 fois l'anglettere, et des pays inexistant comme XX, QQ et ZZ.

In [7]:
display(df.pays_declare.unique())

<StringArray>
['DK', 'PT', 'LU', 'SE', 'NL', 'BE', 'FI', 'GB', 'FR', 'ZZ', 'PL', 'IT', 'UK',
 'XX', 'QQ']
Length: 15, dtype: str

- Quelle proportion de lignes porte des caractères qui n'ont rien à faire dans un numéro de TVA ?

In [8]:
mask = df["numero_tva"].str.contains(r"[^A-Za-z0-9]", na=False)
proportion = mask.mean()
print(f"{proportion:.2%} des lignes ({mask.sum()} sur {len(df)})")

18.13% des lignes (1813 sur 10000)


Combien de valeurs vides, et sous combien de formes différentes le vide se présente-t-il ?

In [9]:
# Comptage simple des différentes formes de vide
vide  = (df["numero_tva"] == "").sum()
espace   = (df["numero_tva"] == " ").sum()
nan     = (df["numero_tva"].astype(str).str.lower() == "nan").sum()
null    = (df["numero_tva"].astype(str).str.lower() == "null").sum()
na      = (df["numero_tva"].astype(str).str.upper() == "N/A").sum()
tiret  = (df["numero_tva"] == "-").sum()
nan     = df["numero_tva"].isna().sum()
print(f'""          : {vide}')
print(f'" "         : {espace}')
print(f'"nan"       : {nan}')
print(f'"null"      : {null}')
print(f'"N/A"       : {na}')
print(f'"-"         : {tiret}')
print(f'NaN (pandas): {nan}')

""          : 0
" "         : 59
"nan"       : 146
"null"      : 0
"N/A"       : 0
"-"         : 55
NaN (pandas): 146


Le pays déclaré dans le CRM correspond-il toujours au préfixe du numéro ?
    - il 1645 valeur sans prefix
    - il y a 8355 valeur avec prefix

In [10]:
prefix = df["numero_tva"] = df["numero_tva"].astype(str).str[:2]
df["compare"] = (df["pays_declare"] == prefix)

print(df["compare"])


0       True
1       True
2       True
3       True
4       True
        ... 
9995    True
9996    True
9997    True
9998    True
9999    True
Name: compare, Length: 10000, dtype: bool


In [11]:
print(df[df['compare'] == False].shape)
print(df[df['compare'] == True].shape)


(1645, 7)
(8355, 7)


Le CSV et l'Excel contiennent-ils vraiment la même chose une fois chargés ? Lequel prenez-vous comme source de vérité, et pourquoi ?

In [12]:
data_csv = pd.read_csv("../data/numeros_tva.csv", dtype=str, keep_default_na=False)
data_xlsx = pd.read_excel("../data/numeros_tva.xlsx", dtype=str, keep_default_na=False)
for col in data_csv.columns:
    differences = data_csv[col] != data_xlsx[col]
    nb_diff = differences.sum()
    print(f"Colonne '{col}' : {nb_diff} différences")
    if nb_diff > 0:
        comparaison = pd.DataFrame({
            "id": data_csv.loc[differences, "id"],
            "CSV": data_csv.loc[differences, col],
            "Excel": data_xlsx.loc[differences, col]
        })
        print(comparaison.head(5))
        print("-" * 40)

Colonne 'id' : 0 différences
Colonne 'raison_sociale' : 0 différences
Colonne 'pays_declare' : 0 différences
Colonne 'numero_tva' : 0 différences
Colonne 'date_saisie' : 0 différences
Colonne 'source_saisie' : 0 différences


Les canaux de saisie et les périodes de saisie se ressemblent-ils, ou certains sortent-ils du lot ?

faire un timestamp 


In [13]:
# df["source_saisie"].value_counts()

display(df["source_saisie"].value_counts())
date = df["date_saisie"].str[0:7].value_counts()
print(date.sort_values(ascending=False))


source_saisie
reprise_erp           2071
crm                   2019
portail_client        1990
saisie_manuelle       1976
import_fournisseur    1944
Name: count, dtype: int64

date_saisie
2024-04    510
2025-08    508
2024-03    504
2024-06    503
2025-07    500
2025-01    497
2024-12    494
2024-11    494
2024-05    489
2024-08    489
2024-07    469
2025-06    468
2025-04    465
2024-01    461
2024-10    458
2025-05    457
2025-03    456
2024-02    446
2025-09    445
2024-09    443
2025-02    428
2025-10     16
Name: count, dtype: int64
